# LDC LPI Plot-Year Composition Processing

This notebook starts from:

- the full georeferenced LPI tall table, and
- the finalized 10,960-code species / functional-group master dictionary.

It produces **two observation scopes**, each at **two ecological resolutions**:

| Scope | Species composition | Functional-group composition |
|---|---|---|
| **Top-hit only** | one top-canopy species per pin at most | one top-canopy FG per pin at most |
| **Multilayer / any-hit** | species presence anywhere on a pin | FG presence anywhere on a pin |

## Definitions

### Top-hit only
Uses **`layer == "TopCanopy"` only**.

- `__NO_CANOPY__` is retained in the raw table but is not vegetation.
- Species/FG numerator = number of unique pins whose TopCanopy code resolves to that plant taxon / group.
- Denominator = all sampled pins in the plot visit.
- Summed plant cover is therefore ≤ 100%.

### Multilayer / any-hit
Uses plant contacts from:

- `TopCanopy`
- `Lower1` … `Lower7`
- `SoilSurface` when the code resolves to a plant (basal hit)

Within a pin, repeated occurrences of the same species or same functional group count **once**.

- A species can have at most 100% cover.
- A functional group can have at most 100% cover.
- Summed multilayer cover across species or groups may exceed 100% because multiple vegetation entities can occur vertically at the same pin.

## Output organization

Outputs are batched **by year**, not by ecological state.

Year batching is preferable here because state assignment is downstream of these vegetation compositions; batching by state would make the response database depend on a later classification.

For every year, four CSV matrices are written:

1. `top_hit/species/LPI_top_hit_species_<YEAR>.csv`
2. `top_hit/functional_group/LPI_top_hit_functional_group_<YEAR>.csv`
3. `multilayer/species/LPI_multilayer_species_<YEAR>.csv`
4. `multilayer/functional_group/LPI_multilayer_functional_group_<YEAR>.csv`

Each CSV has one row per `PrimaryKey × Year`, metadata columns first, then percent-cover columns.


In [2]:
# ============================================================================
# 1. IMPORTS AND PATHS
# ============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

try:
    import duckdb
except ImportError:
    raise ImportError(
        "DuckDB is required for the 16M-row processing step.\n"
        "Run: %pip install duckdb"
    )

BASE_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
)

RAW_LPI_FILE = (
    BASE_DIR /
    "LDC_LPI_georeferenced_2018_present.csv"
)

FINAL_MASTER_FILE = (
    BASE_DIR /
    "species_dictionary_outputs" /
    "LDC_LPI_species_dictionary_FINAL_MASTER.csv"
)

OUTPUT_DIR = (
    BASE_DIR /
    "processed_LPI_composition"
)

TOP_SPECIES_DIR = OUTPUT_DIR / "top_hit" / "species"
TOP_FG_DIR = OUTPUT_DIR / "top_hit" / "functional_group"
MULTI_SPECIES_DIR = OUTPUT_DIR / "multilayer" / "species"
MULTI_FG_DIR = OUTPUT_DIR / "multilayer" / "functional_group"
QA_DIR = OUTPUT_DIR / "QA"

for p in [
    TOP_SPECIES_DIR,
    TOP_FG_DIR,
    MULTI_SPECIES_DIR,
    MULTI_FG_DIR,
    QA_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

DUCKDB_FILE = OUTPUT_DIR / "LDC_LPI_composition_processing.duckdb"

assert RAW_LPI_FILE.exists(), RAW_LPI_FILE
assert FINAL_MASTER_FILE.exists(), FINAL_MASTER_FILE

print("Raw LPI:")
print(RAW_LPI_FILE)

print("\nFinal master:")
print(FINAL_MASTER_FILE)

print("\nOutput root:")
print(OUTPUT_DIR)


Raw LPI:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\LDC_LPI_georeferenced_2018_present.csv

Final master:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\LDC_LPI_species_dictionary_FINAL_MASTER.csv

Output root:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\processed_LPI_composition


In [3]:
# ============================================================================
# 2. FINAL MASTER QA
# ============================================================================

master = pd.read_csv(
    FINAL_MASTER_FILE,
    low_memory=False
)

assert len(master) == 10960
assert master["observed_code"].nunique(dropna=False) == 10960
assert not master["observed_code"].duplicated().any()

assert master["observed_code"].eq("__NO_CANOPY__").sum() == 1
assert master["observed_code"].eq("NONE").sum() == 1

poar = master.loc[
    master["observed_code"].eq("POAR2R2")
]

assert len(poar) == 1
assert poar.iloc[0]["USDA_accepted_symbol"] == "PONIN"

print("Final master QA: PASS")


Final master QA: PASS


In [4]:
# ============================================================================
# 3. OPEN DUCKDB AND LOAD LOOKUP
# ============================================================================

def sql_path(path: Path) -> str:
    return path.as_posix().replace("'", "''")

raw_sql = sql_path(RAW_LPI_FILE)
master_sql = sql_path(FINAL_MASTER_FILE)

con = duckdb.connect(str(DUCKDB_FILE))
con.execute("PRAGMA threads=8")

con.execute(f"""
CREATE OR REPLACE TABLE species_master AS
SELECT *
FROM read_csv_auto(
    '{master_sql}',
    header = TRUE,
    all_varchar = TRUE
);
""")

qa = con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT observed_code) AS n_unique
FROM species_master
""").df()

display(qa)

assert qa.loc[0, "n_rows"] == 10960
assert qa.loc[0, "n_unique"] == 10960


,n_rows,n_unique
0,10960,10960


## Canonical joined tall table

The raw LPI code is preserved as `observed_code_raw`.

Blank/NA source codes become `__NO_CANOPY__`. Literal `"NONE"` remains a real USDA plant symbol.

For plant rows, `taxon_id` uses the accepted USDA symbol when one exists; otherwise the canonical observed plant code is retained.


In [8]:
# ============================================================================
# DIAGNOSTIC — INSPECT ACTUAL INPUT SCHEMAS BEFORE JOINING
# ============================================================================

print("=" * 80)
print("FINAL MASTER SCHEMA")
print("=" * 80)

master_schema = con.execute("""
DESCRIBE species_master
""").df()

display(master_schema)

master_columns = master_schema["column_name"].astype(str).tolist()

print("\nFinal master columns:")
for c in master_columns:
    print(c)


print("\n" + "=" * 80)
print("RAW GEOREFERENCED LPI SCHEMA")
print("=" * 80)

raw_schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_csv_auto(
    '{raw_sql}',
    header = TRUE,
    all_varchar = TRUE
)
""").df()

display(raw_schema)

raw_columns = raw_schema["column_name"].astype(str).tolist()

print("\nRaw LPI columns:")
for c in raw_columns:
    print(c)


print("\n" + "=" * 80)
print("CRITICAL FINAL-MASTER FIELDS")
print("=" * 80)

critical_master = [
    "observed_code",
    "resolved_label",
    "code_class",
    "resolution_source",
    "USDA_symbol",
    "USDA_accepted_symbol",
    "USDA_scientific_name",
    "USDA_common_name",
    "USDA_family",
    "USDA_rank",
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status",
    "USDA_trait_status",
    "MOSAIC_FG",
    "FG_resolution_source",
    "requires_row_level_resolution",
    "context_protocol_label",
    "context_protocol_class",
    "context_protocol_rule",
]

for c in critical_master:
    print(
        f"{c:35s}",
        "YES" if c in master_columns else "NO"
    )


print("\n" + "=" * 80)
print("FINAL MASTER BASIC INTEGRITY")
print("=" * 80)

master_integrity = con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT observed_code) AS n_unique_codes,
    SUM(CASE WHEN observed_code IS NULL THEN 1 ELSE 0 END) AS null_codes,
    SUM(CASE WHEN code_class IS NULL THEN 1 ELSE 0 END) AS null_code_class,
    SUM(CASE WHEN MOSAIC_FG IS NULL THEN 1 ELSE 0 END) AS null_fg
FROM species_master
""").df()

display(master_integrity)


print("\nSentinel rows:")

sentinels = con.execute("""
SELECT *
FROM species_master
WHERE observed_code IN (
    '__NO_CANOPY__',
    'NONE',
    'POAR2R2',
    'BAPRV',
    'BAPRG'
)
ORDER BY observed_code
""").df()

display(sentinels)

FINAL MASTER SCHEMA


,column_name,column_type,null,key,default,extra
0,code,VARCHAR,YES,None,None,None
1,n_records,VARCHAR,YES,None,None,None
2,n_plot_visits,VARCHAR,YES,None,None,None
3,n_top,VARCHAR,YES,None,None,None
4,n_lower,VARCHAR,YES,None,None,None
5,n_soil_surface,VARCHAR,YES,None,None,None
6,n_top_plots,VARCHAR,YES,None,None,None
7,n_multilayer_plots,VARCHAR,YES,None,None,None
8,first_year,VARCHAR,YES,None,None,None
9,last_year,VARCHAR,YES,None,None,None



Final master columns:
code
n_records
n_plot_visits
n_top
n_lower
n_soil_surface
n_top_plots
n_multilayer_plots
first_year
last_year
observed_code_raw
source_code_was_blank
observed_code
protocol_label
protocol_code_class
code_class
resolved_label
USDA_accepted_symbol
USDA_match_method
USDA_scientific_name
USDA_common_name
USDA_family
protocol_plant_group
generic_code_prefix
resolution_source
taxonomic_level
_years_present_span
USDA_duration
USDA_growth_habit
USDA_native_status_L48
USDA_trait_status
USDA_derived_group
resolved_plant_group
MOSAIC_FG
FG_resolution_source
USDA_symbol_in_trait_cache
USDA_symbol

RAW GEOREFERENCED LPI SCHEMA


,column_name,column_type,null,key,default,extra
0,rid,VARCHAR,YES,None,None,None
1,PrimaryKey,VARCHAR,YES,None,None,None
2,DBKey.x,VARCHAR,YES,None,None,None
3,ProjectKey.x,VARCHAR,YES,None,None,None
4,LineKey,VARCHAR,YES,None,None,None
5,RecKey,VARCHAR,YES,None,None,None
6,layer,VARCHAR,YES,None,None,None
7,code,VARCHAR,YES,None,None,None
8,chckbox,VARCHAR,YES,None,None,None
9,ShrubShape,VARCHAR,YES,None,None,None



Raw LPI columns:
rid
PrimaryKey
DBKey.x
ProjectKey.x
LineKey
RecKey
layer
code
chckbox
ShrubShape
FormType
FormDate
Direction
Measure
LineLengthAmount
SpacingIntervalAmount
SpacingType
ShowCheckbox
CheckboxLabel
PointLoc
PointNbr
source.x
DateLoadedInDb
DateVisited
DateVisited_parsed
SampleYear
Latitude_NAD83
Longitude_NAD83
PlotID
ProjectKey.y
source.y
EcologicalSiteID
DBKey.y

CRITICAL FINAL-MASTER FIELDS
observed_code                       YES
resolved_label                      YES
code_class                          YES
resolution_source                   YES
USDA_symbol                         YES
USDA_accepted_symbol                YES
USDA_scientific_name                YES
USDA_common_name                    YES
USDA_family                         YES
USDA_rank                           NO
USDA_duration                       YES
USDA_growth_habit                   YES
USDA_native_status                  NO
USDA_trait_status                   YES
MOSAIC_FG                     

,n_rows,n_unique_codes,null_codes,null_code_class,null_fg
0,10960,10960,0.0,165.0,209.0



Sentinel rows:


,code,n_records,n_plot_visits,n_top,n_lower,n_soil_surface,n_top_plots,n_multilayer_plots,first_year,last_year,...,USDA_duration,USDA_growth_habit,USDA_native_status_L48,USDA_trait_status,USDA_derived_group,resolved_plant_group,MOSAIC_FG,FG_resolution_source,USDA_symbol_in_trait_cache,USDA_symbol
0,BAPRG,4,2,3,1,0,1,2,2021,2021,...,None,None,None,No_USDA_taxon,None,Forb,ExoticForb,manual_ecological_functional_override,False,NaN
1,BAPRV,14,4,13,1,0,4,4,2021,2021,...,None,None,None,Missing_duration_growth_habit_nativity,None,Forb,ExoticForb,manual_ecological_functional_override,False,NaN
2,NONE,398,13,394,4,0,10,13,2018,2019,...,None,None,None,Missing_duration_growth_habit_nativity,None,NaN,TraitUnresolvedPlant,USDA_taxon_missing_traits,False,NONE
3,POAR2R2,10,1,3,7,0,1,1,2021,2021,...,None,None,None,Missing_duration_growth_habit_nativity,None,NaN,TraitUnresolvedPlant,USDA_taxon_missing_traits,False,POAR2R2
4,NaN,442612,8652,442612,0,0,8652,8652,2018,2023,...,None,None,None,No_USDA_taxon,None,NaN,NO_CANOPY,NaN,False,NaN


In [7]:
# ============================================================================
# 4. JOIN FULL TALL LPI TO FINAL MASTER
# ============================================================================

print("Inspecting raw georeferenced LPI schema...")

raw_schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_csv_auto(
    '{raw_sql}',
    header = TRUE,
    all_varchar = TRUE
)
""").df()

raw_columns = set(
    raw_schema["column_name"]
    .astype(str)
)

print("Columns found:")
print(sorted(raw_columns))


# ============================================================================
# REQUIRED RAW FIELDS
#
# Only fields actually required for:
#   - pin identity
#   - layer structure
#   - code resolution
#   - year
#   - coordinates
# ============================================================================

required_raw_cols = {
    "rid",
    "PrimaryKey",
    "LineKey",
    "RecKey",
    "layer",
    "PointNbr",
    "code",
    "DateVisited",
    "Latitude_NAD83",
    "Longitude_NAD83",
}

missing_required = sorted(
    required_raw_cols - raw_columns
)

assert not missing_required, (
    "Required columns missing from georeferenced LPI CSV: "
    + ", ".join(missing_required)
)


# ============================================================================
# OPTIONAL PROVENANCE FIELDS
#
# These may have existed upstream but are not required for cover calculation.
# NULL placeholders keep a stable downstream schema.
# ============================================================================

dbkey_sql = (
    "DBKey"
    if "DBKey" in raw_columns
    else "NULL::VARCHAR AS DBKey"
)

projectkey_sql = (
    "ProjectKey"
    if "ProjectKey" in raw_columns
    else "NULL::VARCHAR AS ProjectKey"
)

source_sql = (
    "source"
    if "source" in raw_columns
    else "NULL::VARCHAR AS source"
)

print("\nOptional provenance fields:")
print("DBKey present:", "DBKey" in raw_columns)
print("ProjectKey present:", "ProjectKey" in raw_columns)
print("source present:", "source" in raw_columns)


# ============================================================================
# BUILD JOINED TALL LPI TABLE
# ============================================================================

print("\nBuilding joined tall LPI table...")

con.execute(f"""
CREATE OR REPLACE TABLE lpi AS

WITH raw AS (

    SELECT

        rid,
        PrimaryKey,

        {dbkey_sql},
        {projectkey_sql},

        LineKey,
        RecKey,
        layer,
        PointNbr,

        -- ----------------------------------------------------
        -- Preserve original source code
        -- ----------------------------------------------------

        code AS observed_code_raw,

        -- ----------------------------------------------------
        -- Canonical code
        --
        -- Blank source code becomes __NO_CANOPY__.
        -- Literal USDA code NONE remains NONE.
        -- ----------------------------------------------------

        CASE
            WHEN code IS NULL
              OR TRIM(code) = ''
            THEN '__NO_CANOPY__'

            ELSE TRIM(code)

        END AS observed_code,

        {source_sql},

        DateVisited,

        -- ----------------------------------------------------
        -- Visit year
        -- ----------------------------------------------------

        COALESCE(

            YEAR(
                TRY_CAST(
                    DateVisited AS TIMESTAMP
                )
            ),

            TRY_CAST(
                SUBSTR(
                    DateVisited,
                    1,
                    4
                )
                AS INTEGER
            )

        ) AS Year,

        -- ----------------------------------------------------
        -- Coordinates
        -- ----------------------------------------------------

        TRY_CAST(
            Latitude_NAD83 AS DOUBLE
        ) AS Latitude_NAD83,

        TRY_CAST(
            Longitude_NAD83 AS DOUBLE
        ) AS Longitude_NAD83

    FROM read_csv_auto(
        '{raw_sql}',
        header = TRUE,
        all_varchar = TRUE
    )
)

SELECT

    r.*,

    -- ========================================================
    -- FINAL MASTER INTERPRETATION
    -- ========================================================

    m.resolved_label,
    m.code_class,
    m.resolution_source,

    m.USDA_accepted_symbol,
    m.USDA_scientific_name,
    m.USDA_rank,

    m.MOSAIC_FG,
    m.FG_resolution_source,

    m.requires_row_level_resolution,
    m.context_protocol_label,
    m.context_protocol_class,
    m.context_protocol_rule,

    -- ========================================================
    -- TAXON ID
    -- ========================================================

    CASE

        WHEN m.code_class = 'Plant'

        THEN COALESCE(

            NULLIF(
                TRIM(
                    m.USDA_accepted_symbol
                ),
                ''
            ),

            r.observed_code
        )

        ELSE NULL

    END AS taxon_id,

    -- ========================================================
    -- HUMAN-READABLE TAXON LABEL
    -- ========================================================

    CASE

        WHEN m.code_class = 'Plant'

        THEN COALESCE(

            NULLIF(
                TRIM(
                    m.USDA_scientific_name
                ),
                ''
            ),

            NULLIF(
                TRIM(
                    m.resolved_label
                ),
                ''
            ),

            r.observed_code
        )

        ELSE NULL

    END AS taxon_label

FROM raw r

LEFT JOIN species_master m

    ON r.observed_code
     = m.observed_code
;
""")


# ============================================================================
# JOIN INTEGRITY QA
# ============================================================================

join_qa = con.execute("""
SELECT

    COUNT(*) AS n_rows,

    SUM(
        CASE
            WHEN code_class IS NULL
            THEN 1
            ELSE 0
        END
    ) AS unmatched_rows,

    COUNT(
        DISTINCT observed_code
    ) AS observed_codes,

    COUNT(
        DISTINCT CASE
            WHEN code_class IS NULL
            THEN observed_code
        END
    ) AS unmatched_codes,

    COUNT(
        DISTINCT PrimaryKey
    ) AS n_primarykeys

FROM lpi
""").df()

print("\nJoin QA:")
display(join_qa)


assert (
    join_qa.loc[
        0,
        "unmatched_rows"
    ]
    == 0
), (
    "One or more LPI rows failed to join to FINAL_MASTER."
)

assert (
    join_qa.loc[
        0,
        "unmatched_codes"
    ]
    == 0
), (
    "One or more canonical observed codes are absent "
    "from FINAL_MASTER."
)


# ============================================================================
# SENTINEL QA
# ============================================================================

sentinel_qa = con.execute("""
SELECT

    observed_code,
    code_class,

    COUNT(*) AS n_records

FROM lpi

WHERE observed_code IN (
    '__NO_CANOPY__',
    'NONE'
)

GROUP BY
    observed_code,
    code_class

ORDER BY
    observed_code
""").df()

print("\nSentinel QA:")
display(sentinel_qa)


# ============================================================================
# YEAR QA
# ============================================================================

year_qa = con.execute("""
SELECT

    Year,

    COUNT(
        DISTINCT PrimaryKey
    ) AS n_plot_visits,

    COUNT(*) AS n_lpi_rows

FROM lpi

GROUP BY Year

ORDER BY Year
""").df()

print("\nYear distribution:")
display(year_qa)


print("\nDictionary join: PASS")

Inspecting raw georeferenced LPI schema...
Columns found:
['CheckboxLabel', 'DBKey.x', 'DBKey.y', 'DateLoadedInDb', 'DateVisited', 'DateVisited_parsed', 'Direction', 'EcologicalSiteID', 'FormDate', 'FormType', 'Latitude_NAD83', 'LineKey', 'LineLengthAmount', 'Longitude_NAD83', 'Measure', 'PlotID', 'PointLoc', 'PointNbr', 'PrimaryKey', 'ProjectKey.x', 'ProjectKey.y', 'RecKey', 'SampleYear', 'ShowCheckbox', 'ShrubShape', 'SpacingIntervalAmount', 'SpacingType', 'chckbox', 'code', 'layer', 'rid', 'source.x', 'source.y']

Optional provenance fields:
DBKey present: False
ProjectKey present: False
source present: False

Building joined tall LPI table...


BinderException: Binder Error: Table "m" does not have a column named "USDA_rank"

Candidate bindings: : "USDA_duration", "USDA_trait_status"

LINE 101:     m.USDA_rank,
              ^

In [9]:
# ============================================================================
# FINAL MASTER — USDA TRAIT COVERAGE AUDIT
#
# Question:
#   Did the repaired USDA trait cache actually propagate into FINAL_MASTER?
#
# Reports coverage:
#   1. across all dictionary codes
#   2. across Plant codes only
#   3. weighted by raw LPI record abundance
#   4. among USDA-resolved plant taxa specifically
# ============================================================================

print("=" * 80)
print("USDA TRAIT COVERAGE AUDIT")
print("=" * 80)


# ============================================================================
# 1. OVERALL PLANT TRAIT COVERAGE
# ============================================================================

trait_coverage = con.execute("""
SELECT

    COUNT(*) AS plant_codes,

    SUM(
        TRY_CAST(n_records AS BIGINT)
    ) AS plant_records,


    -- ----------------------------------------------------------------------
    -- Duration
    -- ----------------------------------------------------------------------

    SUM(
        CASE
            WHEN USDA_duration IS NOT NULL
             AND TRIM(USDA_duration) <> ''
            THEN 1 ELSE 0
        END
    ) AS codes_with_duration,

    SUM(
        CASE
            WHEN USDA_duration IS NOT NULL
             AND TRIM(USDA_duration) <> ''
            THEN TRY_CAST(n_records AS BIGINT)
            ELSE 0
        END
    ) AS records_with_duration,


    -- ----------------------------------------------------------------------
    -- Growth habit
    -- ----------------------------------------------------------------------

    SUM(
        CASE
            WHEN USDA_growth_habit IS NOT NULL
             AND TRIM(USDA_growth_habit) <> ''
            THEN 1 ELSE 0
        END
    ) AS codes_with_growth_habit,

    SUM(
        CASE
            WHEN USDA_growth_habit IS NOT NULL
             AND TRIM(USDA_growth_habit) <> ''
            THEN TRY_CAST(n_records AS BIGINT)
            ELSE 0
        END
    ) AS records_with_growth_habit,


    -- ----------------------------------------------------------------------
    -- L48 nativity
    -- ----------------------------------------------------------------------

    SUM(
        CASE
            WHEN USDA_native_status_L48 IS NOT NULL
             AND TRIM(USDA_native_status_L48) <> ''
            THEN 1 ELSE 0
        END
    ) AS codes_with_nativity,

    SUM(
        CASE
            WHEN USDA_native_status_L48 IS NOT NULL
             AND TRIM(USDA_native_status_L48) <> ''
            THEN TRY_CAST(n_records AS BIGINT)
            ELSE 0
        END
    ) AS records_with_nativity,


    -- ----------------------------------------------------------------------
    -- All three
    -- ----------------------------------------------------------------------

    SUM(
        CASE
            WHEN USDA_duration IS NOT NULL
             AND TRIM(USDA_duration) <> ''

             AND USDA_growth_habit IS NOT NULL
             AND TRIM(USDA_growth_habit) <> ''

             AND USDA_native_status_L48 IS NOT NULL
             AND TRIM(USDA_native_status_L48) <> ''

            THEN 1 ELSE 0
        END
    ) AS codes_with_all_three,

    SUM(
        CASE
            WHEN USDA_duration IS NOT NULL
             AND TRIM(USDA_duration) <> ''

             AND USDA_growth_habit IS NOT NULL
             AND TRIM(USDA_growth_habit) <> ''

             AND USDA_native_status_L48 IS NOT NULL
             AND TRIM(USDA_native_status_L48) <> ''

            THEN TRY_CAST(n_records AS BIGINT)
            ELSE 0
        END
    ) AS records_with_all_three

FROM species_master

WHERE code_class = 'Plant'
""").df()

display(trait_coverage)


# ============================================================================
# 2. TURN INTO PERCENTAGES
# ============================================================================

x = trait_coverage.iloc[0]

coverage_pct = pd.DataFrame({

    "trait": [
        "Duration",
        "Growth habit",
        "L48 nativity",
        "All three",
    ],

    "code_coverage_pct": [
        100 * x["codes_with_duration"] / x["plant_codes"],
        100 * x["codes_with_growth_habit"] / x["plant_codes"],
        100 * x["codes_with_nativity"] / x["plant_codes"],
        100 * x["codes_with_all_three"] / x["plant_codes"],
    ],

    "record_weighted_coverage_pct": [
        100 * x["records_with_duration"] / x["plant_records"],
        100 * x["records_with_growth_habit"] / x["plant_records"],
        100 * x["records_with_nativity"] / x["plant_records"],
        100 * x["records_with_all_three"] / x["plant_records"],
    ],
})

print("\nPlant trait coverage:")
display(
    coverage_pct.round(3)
)


# ============================================================================
# 3. USDA-RESOLVED PLANTS ONLY
#
# This is the crucial diagnostic.
#
# If a plant has an accepted USDA symbol but traits are mostly missing,
# something went wrong in the trait-cache merge.
# ============================================================================

usda_resolved_coverage = con.execute("""
SELECT

    COUNT(*) AS USDA_resolved_plant_codes,

    SUM(
        TRY_CAST(n_records AS BIGINT)
    ) AS USDA_resolved_plant_records,

    SUM(
        CASE
            WHEN USDA_duration IS NOT NULL
             AND TRIM(USDA_duration) <> ''
            THEN 1 ELSE 0
        END
    ) AS duration_codes,

    SUM(
        CASE
            WHEN USDA_growth_habit IS NOT NULL
             AND TRIM(USDA_growth_habit) <> ''
            THEN 1 ELSE 0
        END
    ) AS growth_habit_codes,

    SUM(
        CASE
            WHEN USDA_native_status_L48 IS NOT NULL
             AND TRIM(USDA_native_status_L48) <> ''
            THEN 1 ELSE 0
        END
    ) AS nativity_codes,

    SUM(
        CASE
            WHEN USDA_duration IS NOT NULL
             AND TRIM(USDA_duration) <> ''
             AND USDA_growth_habit IS NOT NULL
             AND TRIM(USDA_growth_habit) <> ''
             AND USDA_native_status_L48 IS NOT NULL
             AND TRIM(USDA_native_status_L48) <> ''
            THEN 1 ELSE 0
        END
    ) AS complete_trait_codes

FROM species_master

WHERE
    code_class = 'Plant'

    AND USDA_accepted_symbol IS NOT NULL
    AND TRIM(USDA_accepted_symbol) <> ''
""").df()

print("\nUSDA-resolved plants:")
display(usda_resolved_coverage)


# ============================================================================
# 4. TRAIT STATUS DISTRIBUTION
# ============================================================================

trait_status = con.execute("""
SELECT

    COALESCE(
        USDA_trait_status,
        '<NULL>'
    ) AS USDA_trait_status,

    COUNT(*) AS n_codes,

    SUM(
        TRY_CAST(n_records AS BIGINT)
    ) AS n_records

FROM species_master

WHERE code_class = 'Plant'

GROUP BY
    COALESCE(
        USDA_trait_status,
        '<NULL>'
    )

ORDER BY
    n_records DESC
""").df()

print("\nUSDA trait-status distribution:")
display(trait_status)


# ============================================================================
# 5. HIGH-ABUNDANCE USDA TAXA MISSING TRAITS
#
# These are the records we care about most.
# ============================================================================

missing_traits = con.execute("""
SELECT

    observed_code,
    USDA_accepted_symbol,
    USDA_scientific_name,

    n_records,

    USDA_duration,
    USDA_growth_habit,
    USDA_native_status_L48,
    USDA_trait_status,

    MOSAIC_FG,
    FG_resolution_source

FROM species_master

WHERE
    code_class = 'Plant'

    AND USDA_accepted_symbol IS NOT NULL
    AND TRIM(USDA_accepted_symbol) <> ''

    AND (
           USDA_duration IS NULL
        OR TRIM(USDA_duration) = ''

        OR USDA_growth_habit IS NULL
        OR TRIM(USDA_growth_habit) = ''

        OR USDA_native_status_L48 IS NULL
        OR TRIM(USDA_native_status_L48) = ''
    )

ORDER BY
    TRY_CAST(n_records AS BIGINT) DESC
""").df()

print(
    "\nUSDA-resolved plant codes missing ≥1 trait:",
    f"{len(missing_traits):,}"
)

display(
    missing_traits.head(50)
)


# ============================================================================
# 6. KNOWN COMMON TAXA SPOT CHECK
# ============================================================================

common_check = con.execute("""
SELECT

    observed_code,
    USDA_accepted_symbol,
    USDA_scientific_name,

    USDA_duration,
    USDA_growth_habit,
    USDA_native_status_L48,
    USDA_trait_status,

    MOSAIC_FG,
    FG_resolution_source,

    n_records

FROM species_master

WHERE observed_code IN (
    'BRTE',
    'POSE',
    'ARTRW8',
    'HECO26',
    'FEID',
    'ELEL5',
    'CHVI8',
    'AGCR'
)

ORDER BY
    TRY_CAST(n_records AS BIGINT) DESC
""").df()

print("\nCommon-taxon spot check:")
display(common_check)

USDA TRAIT COVERAGE AUDIT


,plant_codes,plant_records,codes_with_duration,records_with_duration,codes_with_growth_habit,records_with_growth_habit,codes_with_nativity,records_with_nativity,codes_with_all_three,records_with_all_three
0,10749,5436341.0,6803.0,3628070.0,7036.0,3641017.0,6652.0,3616607.0,6650.0,3616554.0



Plant trait coverage:


,trait,code_coverage_pct,record_weighted_coverage_pct
0,Duration,63.290,66.737
1,Growth habit,65.457,66.976
2,L48 nativity,61.885,66.526
3,All three,61.866,66.526



USDA-resolved plants:


...

,USDA_trait_status,n_codes,n_records
0,Complete,6650,3616554.0
1,Missing_duration_growth_habit_nativity,569,1576018.0
2,No_USDA_taxon,3144,219306.0
3,Missing_duration_nativity,231,12894.0
4,Missing_nativity,153,11516.0
5,Missing_duration,2,53.0



USDA-resolved plant codes missing ≥1 trait: 955


,observed_code,USDA_accepted_symbol,USDA_scientific_name,n_records,USDA_duration,USDA_growth_habit,USDA_native_status_L48,USDA_trait_status,MOSAIC_FG,FG_resolution_source
0,PSSP6,PSSP6,Pseudoroegneria spicata (Pursh) Á. Löve,163393,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits
1,POPR,POPR,Poa pratensis L.,115614,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits
2,AGCR,AGCR,Agropyron cristatum (L.) Gaertn.,86637,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits
3,HECO26,HECO26,Hesperostipa comata (Trin. & Rupr.) Barkworth,80033,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits
4,FEID,FEID,Festuca idahoensis Elmer,78604,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits
5,ELEL5,ELEL5,Elymus elymoides (Raf.) Swezey,58454,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits
6,BRIN2,BRIN2,Bromus inermis Leyss.,56601,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits
7,CHVI8,CHVI8,Chrysothamnus viscidiflorus (Hook.) Nutt.,52275,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits
8,SCAR7,SCAR7,"Schedonorus arundinaceus (Schreb.) Dumort., no...",46213,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits
9,ARAR8,ARAR8,Artemisia arbuscula Nutt.,32897,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits



Common-taxon spot check:


,observed_code,USDA_accepted_symbol,USDA_scientific_name,USDA_duration,USDA_growth_habit,USDA_native_status_L48,USDA_trait_status,MOSAIC_FG,FG_resolution_source,n_records
0,BRTE,BRTE,Bromus tectorum L.,Annual,Graminoid,Introduced,Complete,EAG,USDA_traits,429816
1,POSE,POSE,Poa secunda J. Presl,Perennial,Graminoid,Native,Complete,PerennialGraminoid,USDA_traits,332740
2,ARTRW8,ARTRW8,Artemisia tridentata Nutt. ssp. wyomingensis B...,Perennial,Shrub|Tree,Native,Complete,Woody,USDA_traits,232161
3,AGCR,AGCR,Agropyron cristatum (L.) Gaertn.,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits,86637
4,HECO26,HECO26,Hesperostipa comata (Trin. & Rupr.) Barkworth,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits,80033
5,FEID,FEID,Festuca idahoensis Elmer,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits,78604
6,ELEL5,ELEL5,Elymus elymoides (Raf.) Swezey,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits,58454
7,CHVI8,CHVI8,Chrysothamnus viscidiflorus (Hook.) Nutt.,NaN,NaN,NaN,Missing_duration_growth_habit_nativity,TraitUnresolvedPlant,USDA_taxon_missing_traits,52275


In [ ]:
# ============================================================================
# 5. DEFINE UNIQUE PIN-DROP DENOMINATOR
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE pins AS

SELECT DISTINCT
    PrimaryKey,
    Year,
    LineKey,
    PointNbr

FROM lpi

WHERE
    PrimaryKey IS NOT NULL
    AND Year IS NOT NULL
    AND LineKey IS NOT NULL
    AND PointNbr IS NOT NULL
;
""")

con.execute("""
CREATE OR REPLACE TABLE plot_totals AS

SELECT
    PrimaryKey,
    Year,
    COUNT(*) AS n_points

FROM pins

GROUP BY
    PrimaryKey,
    Year
;
""")

point_qa = con.execute("""
SELECT
    COUNT(*) AS n_plot_years,
    MIN(n_points) AS min_points,
    MEDIAN(n_points) AS median_points,
    MAX(n_points) AS max_points,
    AVG(n_points) AS mean_points
FROM plot_totals
""").df()

display(point_qa)


In [ ]:
# ============================================================================
# 6. PLOT-YEAR METADATA
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE plot_metadata AS

SELECT
    p.PrimaryKey,
    p.Year,

    MIN(l.DateVisited) AS DateVisited,
    MIN(l.DBKey) AS DBKey,
    MIN(l.source) AS source,

    MIN(l.Latitude_NAD83) AS Latitude_NAD83,
    MIN(l.Longitude_NAD83) AS Longitude_NAD83,

    p.n_points

FROM plot_totals p

LEFT JOIN lpi l
    ON  p.PrimaryKey = l.PrimaryKey
    AND p.Year = l.Year

GROUP BY
    p.PrimaryKey,
    p.Year,
    p.n_points
;
""")


## Protocol QA for `__NO_CANOPY__`

A `__NO_CANOPY__` TopCanopy observation implies that there are no `Lower1`–`Lower7` records for that pin. The only other observation at that pin should be `SoilSurface`.

This is treated as a protocol invariant and tested explicitly before composition is calculated.


In [ ]:
# ============================================================================
# 6A. QA — NO_CANOPY PINS MUST HAVE NO LOWER-LAYER CONTACTS
# ============================================================================

no_canopy_lower_violations = con.execute("""
WITH no_canopy_pins AS (

    SELECT DISTINCT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr

    FROM lpi

    WHERE
        layer = 'TopCanopy'
        AND observed_code = '__NO_CANOPY__'
)

SELECT
    l.PrimaryKey,
    l.Year,
    l.LineKey,
    l.PointNbr,
    l.layer,
    l.observed_code

FROM lpi l

INNER JOIN no_canopy_pins n
    USING (
        PrimaryKey,
        Year,
        LineKey,
        PointNbr
    )

WHERE l.layer IN (
    'Lower1',
    'Lower2',
    'Lower3',
    'Lower4',
    'Lower5',
    'Lower6',
    'Lower7'
)
""").df()

print(
    "Lower-layer records beneath NO_CANOPY:",
    len(no_canopy_lower_violations)
)

if len(no_canopy_lower_violations):
    display(no_canopy_lower_violations.head(50))

assert len(no_canopy_lower_violations) == 0, (
    "Found lower-layer contacts beneath a __NO_CANOPY__ TopCanopy record. "
    "This violates the expected LPI recording structure and should be "
    "investigated before calculating composition."
)

print("NO_CANOPY protocol QA: PASS")


# A. Top-hit / top-down composition

The top-hit product represents the **top-down outcome at each pin**.

For each pin:

1. If `TopCanopy` contains an actual contact, that record is the top hit.
2. If `TopCanopy == "__NO_CANOPY__"`, then the pin has no lower-layer contacts by protocol and the **`SoilSurface` record becomes the top hit**.

`Lower1`–`Lower7` are never used to replace a `__NO_CANOPY__` observation.

The species and functional-group top-hit outputs remain **vegetation fractional cover only**:

- numerator = pins whose derived top hit is a plant taxon / functional group,
- denominator = **all sampled pins**.

Therefore a no-canopy pin whose SoilSurface is soil, litter, rock, lichen, etc. contributes zero to every plant species/FG cover while remaining in the denominator. This preserves true fractional cover rather than renormalizing only across vegetated pins.

A separate top-hit surface-composition QA table is also produced so the nonvegetated portion remains visible.


In [ ]:
# ============================================================================
# 7. DERIVE TOP HIT PER PIN + TOP-HIT SPECIES COVER
#
# Protocol:
#   - actual TopCanopy contact wins
#   - __NO_CANOPY__ falls directly to SoilSurface
#   - Lower1-Lower7 are not part of this derivation
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE lpi_top_hit AS

WITH top AS (

    SELECT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr,

        observed_code,
        code_class,
        taxon_id,
        taxon_label,
        USDA_rank,
        MOSAIC_FG,

        CASE
            WHEN observed_code = '__NO_CANOPY__'
            THEN 1
            ELSE 0
        END AS no_canopy

    FROM lpi

    WHERE layer = 'TopCanopy'
),

surface AS (

    SELECT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr,

        observed_code,
        code_class,
        taxon_id,
        taxon_label,
        USDA_rank,
        MOSAIC_FG

    FROM lpi

    WHERE layer = 'SoilSurface'
)

SELECT

    t.PrimaryKey,
    t.Year,
    t.LineKey,
    t.PointNbr,

    CASE
        WHEN t.no_canopy = 0
        THEN t.observed_code
        ELSE s.observed_code
    END AS observed_code,

    CASE
        WHEN t.no_canopy = 0
        THEN t.code_class
        ELSE s.code_class
    END AS code_class,

    CASE
        WHEN t.no_canopy = 0
        THEN t.taxon_id
        ELSE s.taxon_id
    END AS taxon_id,

    CASE
        WHEN t.no_canopy = 0
        THEN t.taxon_label
        ELSE s.taxon_label
    END AS taxon_label,

    CASE
        WHEN t.no_canopy = 0
        THEN t.USDA_rank
        ELSE s.USDA_rank
    END AS USDA_rank,

    CASE
        WHEN t.no_canopy = 0
        THEN t.MOSAIC_FG
        ELSE s.MOSAIC_FG
    END AS MOSAIC_FG,

    CASE
        WHEN t.no_canopy = 0
        THEN 'TopCanopy'
        ELSE 'SoilSurface'
    END AS top_hit_source

FROM top t

LEFT JOIN surface s
    USING (
        PrimaryKey,
        Year,
        LineKey,
        PointNbr
    )
;
""")

# Every sampled pin should resolve to one top-hit row.
top_hit_qa = con.execute("""
SELECT
    COUNT(*) AS n_top_hit_rows,
    COUNT(
        DISTINCT (
            PrimaryKey,
            Year,
            LineKey,
            PointNbr
        )
    ) AS n_unique_top_hit_pins,
    SUM(
        CASE
            WHEN observed_code IS NULL
            THEN 1
            ELSE 0
        END
    ) AS unresolved_top_hits
FROM lpi_top_hit
""").df()

display(top_hit_qa)

# Species fractional cover, denominator = all sampled pins.
con.execute("""
CREATE OR REPLACE TABLE top_species_long AS

WITH hits AS (

    SELECT
        PrimaryKey,
        Year,
        taxon_id,
        taxon_label,
        USDA_rank,
        COUNT(*) AS n_hit_points

    FROM lpi_top_hit

    WHERE
        code_class = 'Plant'
        AND taxon_id IS NOT NULL

    GROUP BY
        PrimaryKey,
        Year,
        taxon_id,
        taxon_label,
        USDA_rank
)

SELECT
    h.PrimaryKey,
    h.Year,
    h.taxon_id,
    h.taxon_label,
    h.USDA_rank,
    h.n_hit_points,
    p.n_points,
    100.0 * h.n_hit_points / p.n_points AS percent_cover

FROM hits h

JOIN plot_totals p
    USING (PrimaryKey, Year)
;
""")


In [ ]:
# ============================================================================
# 8. TOP-HIT FUNCTIONAL-GROUP COVER + SURFACE-COMPOSITION QA
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE top_fg_long AS

WITH hits AS (

    SELECT
        PrimaryKey,
        Year,
        MOSAIC_FG,
        COUNT(*) AS n_hit_points

    FROM lpi_top_hit

    WHERE
        code_class = 'Plant'
        AND MOSAIC_FG IS NOT NULL

    GROUP BY
        PrimaryKey,
        Year,
        MOSAIC_FG
)

SELECT
    h.PrimaryKey,
    h.Year,
    h.MOSAIC_FG,
    h.n_hit_points,
    p.n_points,
    100.0 * h.n_hit_points / p.n_points AS percent_cover

FROM hits h

JOIN plot_totals p
    USING (PrimaryKey, Year)
;
""")

# Plant FG fractional cover cannot exceed 100% because each pin has one top hit.
top_fg_qa = con.execute("""
SELECT
    PrimaryKey,
    Year,
    SUM(percent_cover) AS summed_top_plant_cover

FROM top_fg_long

GROUP BY
    PrimaryKey,
    Year

HAVING
    SUM(percent_cover) > 100.000001
""").df()

assert len(top_fg_qa) == 0

# Preserve nonvegetated top-down composition for QA / interpretation.
top_surface_composition = con.execute("""
SELECT

    PrimaryKey,
    Year,
    observed_code,
    code_class,
    top_hit_source,

    COUNT(*) AS n_points

FROM lpi_top_hit

GROUP BY
    PrimaryKey,
    Year,
    observed_code,
    code_class,
    top_hit_source

ORDER BY
    Year,
    PrimaryKey,
    n_points DESC
""").df()

TOP_SURFACE_QA_FILE = (
    QA_DIR /
    "LPI_top_hit_surface_composition_long.csv"
)

top_surface_composition.to_csv(
    TOP_SURFACE_QA_FILE,
    index=False
)

top_source_qa = con.execute("""
SELECT
    top_hit_source,
    COUNT(*) AS n_pins,
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER () AS percent_pins
FROM lpi_top_hit
GROUP BY top_hit_source
ORDER BY n_pins DESC
""").df()

display(top_source_qa)

print("Top-hit cover exclusivity QA: PASS")
print("\nTop-hit surface-composition QA:")
print(TOP_SURFACE_QA_FILE)


# B. Multilayer / any-hit

Multilayer includes all plant contacts recorded at a pin:

- `TopCanopy`
- `Lower1`–`Lower7`
- plant-coded `SoilSurface` basal hits

If a pin has `TopCanopy == "__NO_CANOPY__"`, there are no lower-layer contacts by protocol, so only its `SoilSurface` record remains available.

For composition calculations, the same species or same functional group is counted only once per pin.


In [ ]:
# ============================================================================
# 9. MULTILAYER SPECIES COVER — LONG TABLE
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE multi_species_long AS

WITH presence AS (

    SELECT DISTINCT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr,
        taxon_id,
        taxon_label,
        USDA_rank

    FROM lpi

    WHERE
        layer IN (
            'TopCanopy',
            'Lower1',
            'Lower2',
            'Lower3',
            'Lower4',
            'Lower5',
            'Lower6',
            'Lower7',
            'SoilSurface'
        )
        AND code_class = 'Plant'
        AND taxon_id IS NOT NULL
),

hits AS (

    SELECT
        PrimaryKey,
        Year,
        taxon_id,
        taxon_label,
        USDA_rank,
        COUNT(*) AS n_hit_points

    FROM presence

    GROUP BY
        PrimaryKey,
        Year,
        taxon_id,
        taxon_label,
        USDA_rank
)

SELECT
    h.PrimaryKey,
    h.Year,
    h.taxon_id,
    h.taxon_label,
    h.USDA_rank,
    h.n_hit_points,
    p.n_points,
    100.0 * h.n_hit_points / p.n_points AS percent_cover

FROM hits h

JOIN plot_totals p
    USING (PrimaryKey, Year)
;
""")


In [ ]:
# ============================================================================
# 10. MULTILAYER FUNCTIONAL-GROUP COVER — LONG TABLE
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE multi_fg_long AS

WITH presence AS (

    SELECT DISTINCT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr,
        MOSAIC_FG

    FROM lpi

    WHERE
        layer IN (
            'TopCanopy',
            'Lower1',
            'Lower2',
            'Lower3',
            'Lower4',
            'Lower5',
            'Lower6',
            'Lower7',
            'SoilSurface'
        )
        AND code_class = 'Plant'
        AND MOSAIC_FG IS NOT NULL
),

hits AS (

    SELECT
        PrimaryKey,
        Year,
        MOSAIC_FG,
        COUNT(*) AS n_hit_points

    FROM presence

    GROUP BY
        PrimaryKey,
        Year,
        MOSAIC_FG
)

SELECT
    h.PrimaryKey,
    h.Year,
    h.MOSAIC_FG,
    h.n_hit_points,
    p.n_points,
    100.0 * h.n_hit_points / p.n_points AS percent_cover

FROM hits h

JOIN plot_totals p
    USING (PrimaryKey, Year)
;
""")


In [ ]:
# ============================================================================
# 11. CONTEXT-DEPENDENT / UNRESOLVED QA
# ============================================================================

context_qa = con.execute("""
SELECT
    observed_code,
    code_class,
    layer,
    COUNT(*) AS n_records,
    COUNT(DISTINCT PrimaryKey) AS n_plot_visits,

    COUNT(
        DISTINCT (
            PrimaryKey,
            LineKey,
            PointNbr
        )
    ) AS n_points

FROM lpi

WHERE
    code_class IN (
        'ContextDependent',
        'ProtocolUnresolved'
    )

GROUP BY
    observed_code,
    code_class,
    layer

ORDER BY
    n_records DESC
""").df()

display(context_qa)

context_qa.to_csv(
    QA_DIR / "LPI_context_dependent_unresolved.csv",
    index=False
)


# Export wide composition matrices by year

Species matrices are intentionally batched by **year** to prevent one national all-years table from accumulating every taxon ever observed into a single column universe.

Functional-group matrices are also written by year for symmetry and easy pairing with annual Sentinel-2 harmonic representations.

Each output row is one `PrimaryKey × Year`. Missing taxa/groups within a plot are written as `0.0`.


In [ ]:
# ============================================================================
# 12. EXPORT FUNCTION — LONG COVER TABLE -> YEAR-SPECIFIC WIDE CSV
# ============================================================================

META_COLS = [
    "PrimaryKey",
    "Year",
    "DateVisited",
    "DBKey",
    "source",
    "Latitude_NAD83",
    "Longitude_NAD83",
    "n_points",
]

def export_year_wide(
    long_table,
    category_col,
    prefix,
    output_dir,
    filename_stub,
):
    """
    Convert one DuckDB long cover table to one wide CSV per year.

    long_table:
        DuckDB table with PrimaryKey, Year, category_col, percent_cover.

    category_col:
        taxon_id or MOSAIC_FG.

    prefix:
        column prefix such as SP_ or FG_.
    """

    years = [
        int(x[0])
        for x in con.execute(
            f"""
            SELECT DISTINCT Year
            FROM {long_table}
            WHERE Year IS NOT NULL
            ORDER BY Year
            """
        ).fetchall()
    ]

    written = []

    for year in years:

        meta = con.execute(
            f"""
            SELECT *
            FROM plot_metadata
            WHERE Year = {year}
            ORDER BY PrimaryKey
            """
        ).df()

        long_df = con.execute(
            f"""
            SELECT
                PrimaryKey,
                Year,
                {category_col},
                percent_cover
            FROM {long_table}
            WHERE Year = {year}
            """
        ).df()

        if long_df.empty:
            wide = meta.copy()

        else:
            matrix = (
                long_df
                .pivot_table(
                    index=["PrimaryKey", "Year"],
                    columns=category_col,
                    values="percent_cover",
                    aggfunc="first",
                    fill_value=0.0,
                )
                .reset_index()
            )

            category_columns = [
                c for c in matrix.columns
                if c not in ["PrimaryKey", "Year"]
            ]

            rename_map = {
                c: f"{prefix}{c}"
                for c in category_columns
            }

            matrix = matrix.rename(columns=rename_map)

            wide = meta.merge(
                matrix,
                on=["PrimaryKey", "Year"],
                how="left",
                validate="one_to_one",
            )

            cover_cols = [
                c for c in wide.columns
                if c.startswith(prefix)
            ]

            wide[cover_cols] = wide[cover_cols].fillna(0.0)

        assert not wide[
            ["PrimaryKey", "Year"]
        ].duplicated().any()

        outfile = (
            output_dir /
            f"{filename_stub}_{year}.csv"
        )

        wide.to_csv(
            outfile,
            index=False
        )

        written.append({
            "year": year,
            "rows": len(wide),
            "columns": len(wide.columns),
            "file": str(outfile),
        })

        print(
            f"{year}: "
            f"{len(wide):,} rows × "
            f"{len(wide.columns):,} columns -> "
            f"{outfile.name}"
        )

    return pd.DataFrame(written)


In [ ]:
# ============================================================================
# 13. WRITE TOP-HIT SPECIES MATRICES BY YEAR
# ============================================================================

top_species_files = export_year_wide(
    long_table="top_species_long",
    category_col="taxon_id",
    prefix="SP_",
    output_dir=TOP_SPECIES_DIR,
    filename_stub="LPI_top_hit_species",
)

display(top_species_files)


In [ ]:
# ============================================================================
# 14. WRITE TOP-HIT FUNCTIONAL-GROUP MATRICES BY YEAR
# ============================================================================

top_fg_files = export_year_wide(
    long_table="top_fg_long",
    category_col="MOSAIC_FG",
    prefix="FG_",
    output_dir=TOP_FG_DIR,
    filename_stub="LPI_top_hit_functional_group",
)

display(top_fg_files)


In [ ]:
# ============================================================================
# 15. WRITE MULTILAYER SPECIES MATRICES BY YEAR
# ============================================================================

multi_species_files = export_year_wide(
    long_table="multi_species_long",
    category_col="taxon_id",
    prefix="SP_",
    output_dir=MULTI_SPECIES_DIR,
    filename_stub="LPI_multilayer_species",
)

display(multi_species_files)


In [ ]:
# ============================================================================
# 16. WRITE MULTILAYER FUNCTIONAL-GROUP MATRICES BY YEAR
# ============================================================================

multi_fg_files = export_year_wide(
    long_table="multi_fg_long",
    category_col="MOSAIC_FG",
    prefix="FG_",
    output_dir=MULTI_FG_DIR,
    filename_stub="LPI_multilayer_functional_group",
)

display(multi_fg_files)


In [ ]:
# ============================================================================
# 17. CROSS-OUTPUT QA
# ============================================================================

# Every output family should cover the same year universe.
year_sets = {
    "top_species": set(top_species_files["year"]),
    "top_fg": set(top_fg_files["year"]),
    "multi_species": set(multi_species_files["year"]),
    "multi_fg": set(multi_fg_files["year"]),
}

assert (
    year_sets["top_species"]
    == year_sets["top_fg"]
    == year_sets["multi_species"]
    == year_sets["multi_fg"]
)

# Top-hit plant FG sums cannot exceed 100%.
top_sum_qa = con.execute("""
SELECT
    PrimaryKey,
    Year,
    SUM(percent_cover) AS summed_cover
FROM top_fg_long
GROUP BY
    PrimaryKey,
    Year
HAVING SUM(percent_cover) > 100.000001
""").df()

assert len(top_sum_qa) == 0

# Individual multilayer taxa/groups cannot exceed 100%.
multi_species_qa = con.execute("""
SELECT *
FROM multi_species_long
WHERE percent_cover > 100.000001
""").df()

multi_fg_qa = con.execute("""
SELECT *
FROM multi_fg_long
WHERE percent_cover > 100.000001
""").df()

assert len(multi_species_qa) == 0
assert len(multi_fg_qa) == 0

print("Cross-output QA: PASS")

print("\nOutput root:")
print(OUTPUT_DIR)


# Resulting data architecture

For every year, the notebook produces four vegetation composition matrices:

### Top-hit species
`top_hit/species/LPI_top_hit_species_<YEAR>.csv`

Top-hit is the top-down outcome at each pin:

- actual `TopCanopy` contact when present,
- otherwise `SoilSurface` when `TopCanopy == "__NO_CANOPY__"`.

Only plant top hits contribute to species cover, but **all pins remain in the denominator**.

### Top-hit functional groups
`top_hit/functional_group/LPI_top_hit_functional_group_<YEAR>.csv`

Functional-group fractional cover uses the same top-hit definition and all-pin denominator.

### Multilayer species
`multilayer/species/LPI_multilayer_species_<YEAR>.csv`

A species is present at a pin if it occurs in any plant contact across the canopy profile or as a basal plant hit at `SoilSurface`.

### Multilayer functional groups
`multilayer/functional_group/LPI_multilayer_functional_group_<YEAR>.csv`

Functional-group presence is deduplicated directly at the pin level rather than obtained by summing species covers.

### Additional QA
`QA/LPI_top_hit_surface_composition_long.csv`

This preserves the nonvegetated top-down outcomes—soil, litter, rock, lichen, etc.—that occur when a no-canopy pin resolves to its SoilSurface observation.

The separation therefore gives two complementary ecological response spaces:

- **top-hit / top-down fractional cover** — closest to what an overhead optical sensor encounters,
- **multilayer / any-hit composition** — fuller vertically encountered plant community composition.
